# Neo4j Graph Data Science Python Client
* https://neo4j.com/docs/graph-data-science-client/current/

# Setup

In [6]:
!pip install graphdatascience
!pip install neo4j-viz[gds]
!pip install python-dotenv

In [2]:
from dotenv import load_dotenv
from neo4j_viz.gds import from_gds
from graphdatascience import GraphDataScience
import os

load_dotenv()

True

# Getting started

In [3]:
URI = "neo4j://localhost:7687"
auth = ("neo4j", os.getenv('NEO4J_AUTH', default='neo4j'))
db = os.getenv('NEO4J_DB', 'neo4j')

gds = GraphDataScience(URI, auth=auth, database=db)

# Check the installed GDS version on the server
print(gds.server_version())

2.13.9


In [4]:
# Create a minimal example graph
gds.run_cypher(
  """
  // CREATE -> MERGE
  MERGE (m: City {name: "Malmö", t: "Malmö"})
  MERGE (l: City {name: "London", t: "London"})
  MERGE (s: City {name: "San Mateo",t: "San Mateo"})
  MERGE (m)-[:FLY_TO]->(l)
  MERGE (l)-[:FLY_TO]->(m)
  MERGE (l)-[:FLY_TO]->(s)
  MERGE (s)-[:FLY_TO]->(l)
  """
)

""


In [6]:
_graph_name = 'neo4j-offices'
if gds.graph.exists(_graph_name) is not None:
  gds.graph.drop(_graph_name)

# Project the graph into the GDS Graph Catalog
# We call the object representing the projected graph `G_office`
G_office, project_result = gds.graph.project(
    _graph_name,
    "City", 
    "FLY_TO")

print('=== project_result\n', project_result)
G_office

=== project_result
 nodeProjection                {'City': {'properties': {}, 'label': 'City'}}
relationshipProjection    {'FLY_TO': {'orientation': 'NATURAL', 'aggrega...
graphName                                                     neo4j-offices
nodeCount                                                                 6
relationshipCount                                                         8
projectMillis                                                            32
Name: 0, dtype: object


Graph({'graphName': 'neo4j-offices', 'nodeCount': 6, 'relationshipCount': 8, 'database': 'neo4j', 'configuration': {'relationshipProjection': {'FLY_TO': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'FLY_TO', 'properties': {}, 'indexInverse': False}}, 'jobId': '214bf441-649e-491c-923b-e2a51fcc27fd', 'nodeProjection': {'City': {'properties': {}, 'label': 'City'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, 'schema': {'graphProperties': {}, 'relationships': {'FLY_TO': {}}, 'nodes': {'City': {}}}, 'memoryUsage': '290 KiB'})

In [7]:
from neo4j_viz.gds import from_gds

VG = from_gds(gds, G_office)
# VG.render()

In [8]:
# Run the mutate mode of the PageRank algorithm
# mutate_result = gds.pageRank.mutate(G_office, tolerance=0.5, mutateProperty="rank")
# print('=== mutate_result\n', mutate_result)
# G_office

write_result = gds.pageRank.write(G_office, tolerance=0.5, writeProperty="rank")
print('=== write_result\n', write_result)
G_office

=== write_result
 writeMillis                                                              13
nodePropertiesWritten                                                     6
ranIterations                                                             1
didConverge                                                            True
centralityDistribution    {'p99': 0.4049997329711914, 'min': 0.213749885...
postProcessingMillis                                                      1
preProcessingMillis                                                       0
computeMillis                                                            21
configuration             {'maxIterations': 20, 'writeConcurrency': 4, '...
Name: 0, dtype: object


Graph({'graphName': 'neo4j-offices', 'nodeCount': 6, 'relationshipCount': 8, 'database': 'neo4j', 'configuration': {'relationshipProjection': {'FLY_TO': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'FLY_TO', 'properties': {}, 'indexInverse': False}}, 'jobId': '214bf441-649e-491c-923b-e2a51fcc27fd', 'nodeProjection': {'City': {'properties': {}, 'label': 'City'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, 'schema': {'graphProperties': {}, 'relationships': {'FLY_TO': {}}, 'nodes': {'City': {}}}, 'memoryUsage': '290 KiB'})

In [ ]:
# We can inspect the node properties of our projected graph directly
# via the graph object and see that indeed the new property exists

# assert G_office.node_properties("City") == ["rank"]

In [9]:

VG = from_gds(gds, G_office)
# VG.render()

# The graph object

See Also
* https://neo4j.com/docs/graph-data-science/current/management-ops/graph-creation/

## Native projection

In [10]:
gds.run_cypher("""
MERGE (florentin:Person { name: 'Florentin', age: 16 })
MERGE (adam:Person { name: 'Adam', age: 18 })
MERGE (veselin:Person { name: 'Veselin', age: 20, ratings: [5.0] })
MERGE (hobbit:Book { name: 'The Hobbit', isbn: 1234, numberOfPages: 310, ratings: [1.0, 2.0, 3.0, 4.5] })
MERGE (frankenstein:Book { name: 'Frankenstein', isbn: 4242, price: 19.99 })

MERGE (florentin)-[:KNOWS { since: 2010 }]->(adam)
MERGE (florentin)-[:KNOWS { since: 2018 }]->(veselin)
MERGE (florentin)-[:READ { numberOfPages: 4 }]->(hobbit)
MERGE (florentin)-[:READ { numberOfPages: 42 }]->(hobbit)
MERGE (adam)-[:READ { numberOfPages: 30 }]->(hobbit)
MERGE (veselin)-[:READ]->(frankenstein)
""")

_graph_name = 'native-projection-g'
if gds.graph.exists(_graph_name) is not None:
  gds.graph.drop(_graph_name)

G, project_result = gds.graph.project(
    _graph_name,
    [{"Person": {"label": "Person", "properties": ['age', 'ratings']}},
     {"Book": {"properties": ['price', 'ratings']}}],
    ["KNOWS", "READ"],
    # nodeProperties=['ratings']
    )

print('=== project_result\n', project_result)
G

=== project_result
 nodeProjection            {'Person': {'properties': {'age': {'property':...
relationshipProjection    {'READ': {'orientation': 'NATURAL', 'aggregati...
graphName                                               native-projection-g
nodeCount                                                                 5
relationshipCount                                                         6
projectMillis                                                            44
Name: 0, dtype: object


Graph({'graphName': 'native-projection-g', 'nodeCount': 5, 'relationshipCount': 6, 'database': 'neo4j', 'configuration': {'relationshipProjection': {'READ': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'READ', 'properties': {}, 'indexInverse': False}, 'KNOWS': {'orientation': 'NATURAL', 'aggregation': 'DEFAULT', 'type': 'KNOWS', 'properties': {}, 'indexInverse': False}}, 'jobId': 'f6ea08ab-d80b-4f17-a859-fa77e280a1e8', 'nodeProjection': {'Person': {'properties': {'age': {'property': 'age', 'defaultValue': None}, 'ratings': {'property': 'ratings', 'defaultValue': None}}, 'label': 'Person'}, 'Book': {'properties': {'price': {'property': 'price', 'defaultValue': None}, 'ratings': {'property': 'ratings', 'defaultValue': None}}, 'label': 'Book'}}, 'relationshipProperties': {}, 'validateRelationships': False, 'logProgress': True, 'readConcurrency': 4, 'sudo': False, 'nodeProperties': {}}, 'schema': {'graphProperties': {}, 'relationships': {'READ': {}, 'KNOWS': {}}, 'nodes': {

In [11]:
VG = from_gds(gds, G,
              # node_properties=['ratings'],  # 选择的节点属性
              db_node_properties=['name']  # 补充的节点属性: 非属性值
              )

# 自定义图中节点名称
# VG.set_node_captions(property="ratings")
for node in VG.nodes:
  caption = '['+str(node.id) + ']'+(node.properties.get("name") or '') + \
      ':'+str(node.properties.get("ratings") or '')
  node.properties['name'] = caption

# VG.render()

In [136]:
VG.nodes

[Node(id=6, caption='[6]Florentin:', caption_align=None, caption_size=None, size=None, color=Color('#ffdf81', rgb=(255, 223, 129)), pinned=None, x=None, y=None, properties={'age': 16, 'name': 'Florentin', 'labels': ['Person']}),
 Node(id=7, caption='[7]Adam:', caption_align=None, caption_size=None, size=None, color=Color('#ffdf81', rgb=(255, 223, 129)), pinned=None, x=None, y=None, properties={'age': 18, 'name': 'Adam', 'labels': ['Person']}),
 Node(id=8, caption='[8]Veselin:[5.0]', caption_align=None, caption_size=None, size=None, color=Color('#ffdf81', rgb=(255, 223, 129)), pinned=None, x=None, y=None, properties={'age': 20, 'ratings': [5.0], 'name': 'Veselin', 'labels': ['Person']}),
 Node(id=9, caption='[9]The Hobbit:[1.0, 2.0, 3.0, 4.5]', caption_align=None, caption_size=None, size=None, color=Color('#c990c0', rgb=(201, 144, 192)), pinned=None, x=None, y=None, properties={'ratings': [1.0, 2.0, 3.0, 4.5], 'name': 'The Hobbit', 'labels': ['Book']}),
 Node(id=10, caption='[10]Franken

# Datasets
Built-in datasets
- Cora
- Karate club
- IMDB
- LastFM
Datasets from the Open Graph Benchmark
- OGBN graphs: Datasets used for node property prediction.
- OGBL graphs: Datasets used for link property prediction.

# Cleanup